# Part 2 — LLM agent: tool calling with LoRA

Model **Qwen2.5-0.5B-Instruct** · 5 tools with made-up data · template dataset (800 train / 100 test, disjoint phrasings)
· baseline vs LoRA (loss on assistant turns only) · metrics: call validity, call accuracy, abstention, task success · failure analysis.

Runtime: *Runtime → Change runtime type → T4 GPU*. Seed 42 everywhere. Whole notebook ≈ 20–30 min on a T4.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U "transformers>=4.46" peft accelerate datasets
import os, sys
USE_DRIVE = True   # checkpoints + results go to Google Drive so a disconnect is not fatal
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/EN4554/part2_agent'
else:
    OUT = '/content/part2_agent_outputs'
os.makedirs(OUT, exist_ok=True)
os.makedirs('/content/part2_agent', exist_ok=True)
%cd /content/part2_agent
sys.path.insert(0, '/content/part2_agent')
SEED = 42
print('outputs ->', OUT)

## 1. Source modules (tools, data generator, agent loop, evaluation, training)

In [ ]:
%%writefile tools.py
"""Five small tools with made-up data, their JSON schemas, and a dispatcher.

The lookup tools (weather, stock price) return invented values, so the model
cannot answer from memory and must call the tool.
"""
import ast
import datetime as dt
import json
import operator

# ----------------------------------------------------------------------------
# Fake data
# ----------------------------------------------------------------------------
WEATHER = {  # city -> (temperature in Celsius, condition)  -- invented values
    "colombo": (31, "humid and sunny"),
    "kandy": (24, "light rain"),
    "paris": (12, "overcast"),
    "tokyo": (17, "clear"),
    "nairobi": (22, "partly cloudy"),
    "oslo": (-3, "snow"),
    "lima": (19, "foggy"),
    "toronto": (5, "windy"),
    "cairo": (34, "sunny"),
    "sydney": (26, "thunderstorms"),
    "berlin": (8, "drizzle"),
    "mumbai": (29, "hazy"),
}

STOCKS = {  # invented tickers -> price in USD
    "ZORB": 142.37,
    "QLXT": 58.12,
    "NIMB": 311.05,
    "VEXA": 23.90,
    "TORQ": 87.44,
    "PLUM": 4.75,
    "KRYO": 1204.50,
    "SOLN": 66.60,
}

# canonical unit -> aliases
UNIT_ALIASES = {
    "km": ["km", "kilometer", "kilometers", "kilometre", "kilometres", "kms"],
    "mi": ["mi", "mile", "miles"],
    "kg": ["kg", "kilogram", "kilograms", "kgs", "kilo", "kilos"],
    "lb": ["lb", "lbs", "pound", "pounds"],
    "C": ["c", "celsius", "°c", "degc", "centigrade"],
    "F": ["f", "fahrenheit", "°f", "degf"],
    "m": ["m", "meter", "meters", "metre", "metres"],
    "ft": ["ft", "foot", "feet"],
    "L": ["l", "liter", "liters", "litre", "litres"],
    "gal": ["gal", "gallon", "gallons"],
    "cm": ["cm", "centimeter", "centimeters", "centimetre", "centimetres"],
    "in": ["in", "inch", "inches"],
}
_ALIAS_TO_UNIT = {a: u for u, aliases in UNIT_ALIASES.items() for a in aliases}

_LINEAR = {  # (from, to) -> factor
    ("km", "mi"): 0.621371,
    ("kg", "lb"): 2.204623,
    ("m", "ft"): 3.280840,
    ("L", "gal"): 0.264172,
    ("cm", "in"): 0.393701,
}


def canon_unit(u):
    if u is None:
        return None
    return _ALIAS_TO_UNIT.get(str(u).strip().lower())


# ----------------------------------------------------------------------------
# Tools
# ----------------------------------------------------------------------------
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
    ast.USub: operator.neg, ast.UAdd: operator.pos,
}


def _safe_eval(node):
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        left, right = _safe_eval(node.left), _safe_eval(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 100:
            raise ValueError("exponent too large")
        return _OPS[type(node.op)](left, right)
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("unsupported expression")


def eval_expression(expression):
    expr = str(expression).replace("^", "**").replace("×", "*").replace("÷", "/").replace(",", "")
    return _safe_eval(ast.parse(expr, mode="eval"))


def calculator(expression):
    try:
        value = eval_expression(expression)
    except Exception as e:  # noqa: BLE001
        return {"error": f"could not evaluate expression: {e}"}
    return {"expression": str(expression), "result": round(float(value), 4)}


def convert_units(value, from_unit, to_unit):
    try:
        value = float(value)
    except (TypeError, ValueError):
        return {"error": "value must be a number"}
    f, t = canon_unit(from_unit), canon_unit(to_unit)
    if f is None or t is None:
        return {"error": f"unknown unit(s): {from_unit}, {to_unit}"}
    if f == t:
        out = value
    elif (f, t) == ("C", "F"):
        out = value * 9 / 5 + 32
    elif (f, t) == ("F", "C"):
        out = (value - 32) * 5 / 9
    elif (f, t) in _LINEAR:
        out = value * _LINEAR[(f, t)]
    elif (t, f) in _LINEAR:
        out = value / _LINEAR[(t, f)]
    else:
        return {"error": f"cannot convert {from_unit} to {to_unit}"}
    return {"value": value, "from_unit": f, "to_unit": t, "result": round(out, 2)}


def get_weather(city):
    key = str(city).strip().lower()
    if key not in WEATHER:
        return {"error": f"no weather data for '{city}'"}
    temp, cond = WEATHER[key]
    return {"city": key.title(), "temperature_c": temp, "condition": cond}


def get_stock_price(ticker):
    key = str(ticker).strip().upper()
    if key not in STOCKS:
        return {"error": f"unknown ticker '{ticker}'"}
    return {"ticker": key, "price_usd": STOCKS[key]}


_DATE_FORMATS = ["%Y-%m-%d", "%Y/%m/%d", "%d-%m-%Y", "%B %d, %Y", "%b %d, %Y", "%d %B %Y", "%d %b %Y"]


def parse_date(s):
    s = str(s).strip()
    for fmt in _DATE_FORMATS:
        try:
            return dt.datetime.strptime(s, fmt).date()
        except ValueError:
            pass
    return None


def days_between(start_date, end_date):
    a, b = parse_date(start_date), parse_date(end_date)
    if a is None or b is None:
        return {"error": "dates must be in YYYY-MM-DD format"}
    return {"start_date": a.isoformat(), "end_date": b.isoformat(), "days": (b - a).days}


TOOLS = {
    "calculator": calculator,
    "convert_units": convert_units,
    "get_weather": get_weather,
    "get_stock_price": get_stock_price,
    "days_between": days_between,
}

TOOL_SCHEMAS = [
    {"type": "function", "function": {
        "name": "calculator",
        "description": "Evaluate an arithmetic expression with + - * / ** % and parentheses.",
        "parameters": {"type": "object", "properties": {
            "expression": {"type": "string", "description": "Arithmetic expression, e.g. '(12 + 7) * 3'"}},
            "required": ["expression"]}}},
    {"type": "function", "function": {
        "name": "convert_units",
        "description": "Convert a value between units. Supported: km/mi, kg/lb, C/F, m/ft, L/gal, cm/in.",
        "parameters": {"type": "object", "properties": {
            "value": {"type": "number", "description": "The value to convert"},
            "from_unit": {"type": "string", "description": "Source unit, e.g. 'km'"},
            "to_unit": {"type": "string", "description": "Target unit, e.g. 'mi'"}},
            "required": ["value", "from_unit", "to_unit"]}}},
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "Get the current weather (temperature in Celsius and condition) for a city.",
        "parameters": {"type": "object", "properties": {
            "city": {"type": "string", "description": "City name, e.g. 'Paris'"}},
            "required": ["city"]}}},
    {"type": "function", "function": {
        "name": "get_stock_price",
        "description": "Get the latest price in USD of a stock by its ticker symbol.",
        "parameters": {"type": "object", "properties": {
            "ticker": {"type": "string", "description": "Ticker symbol, e.g. 'ZORB'"}},
            "required": ["ticker"]}}},
    {"type": "function", "function": {
        "name": "days_between",
        "description": "Number of days from start_date to end_date. Dates use YYYY-MM-DD format.",
        "parameters": {"type": "object", "properties": {
            "start_date": {"type": "string", "description": "Start date, YYYY-MM-DD"},
            "end_date": {"type": "string", "description": "End date, YYYY-MM-DD"}},
            "required": ["start_date", "end_date"]}}},
]


def execute_tool(name, arguments):
    """Run a tool by name. Always returns a JSON-serialisable dict."""
    if name not in TOOLS:
        return {"error": f"unknown tool '{name}'"}
    if not isinstance(arguments, dict):
        return {"error": "arguments must be an object"}
    try:
        return TOOLS[name](**arguments)
    except TypeError as e:
        return {"error": f"bad arguments: {e}"}


def tool_result_str(result):
    return json.dumps(result, ensure_ascii=False)

In [ ]:
%%writefile make_data.py
"""Template-based dataset generator for the tool-calling agent.

Train and test use *disjoint* phrasing templates (and a different date format
for the date task), so the test set measures generalisation to new wording.

Each record:
  id, split, category, query,
  gold_calls: list of {"name", "arguments"} in order,
  checks: list of {"type": "num"|"str", "value"} the final answer must satisfy,
  messages: full gold conversation (user / assistant tool_calls / tool / assistant)
"""
import argparse
import json
import os
import random

from tools import STOCKS, WEATHER, execute_tool, tool_result_str

SEED = 42
CITIES = [c.title() for c in WEATHER]
TICKERS = list(STOCKS)

# ----------------------------------------------------------------------------
# Phrasing templates: TRAIN and TEST pools never share a template.
# ----------------------------------------------------------------------------
T = {
    "weather": {
        "train": ["What's the weather like in {city} right now?",
                  "How is the weather in {city}?",
                  "Tell me the current weather in {city}.",
                  "Is it hot or cold in {city} today?",
                  "weather {city}",
                  "Can you check the weather for {city}?"],
        "test": ["I'm flying to {city} tonight, what conditions should I expect there?",
                 "Give me a quick weather report for {city}, please.",
                 "Any idea what it's like outside in {city} at the moment?",
                 "Look up the current temperature and sky conditions in {city}."],
    },
    "stock": {
        "train": ["What is the current price of {ticker}?",
                  "How much is {ticker} stock trading at?",
                  "Get me the share price for {ticker}.",
                  "{ticker} price?",
                  "Check the stock price of {ticker} for me."],
        "test": ["What's one share of {ticker} going for these days?",
                 "I want to know the latest quote on the {ticker} ticker.",
                 "Could you pull up where {ticker} is trading?",
                 "Before I invest, tell me what {ticker} costs per share."],
    },
    "calc": {
        "train": [("What is {a} times {b}?", "{a} * {b}"),
                  ("Calculate {a} plus {b}.", "{a} + {b}"),
                  ("What's {a} minus {b}?", "{a} - {b}"),
                  ("Divide {a} by {b}.", "{a} / {b}"),
                  ("Compute ({a} + {b}) * {c}.", "({a} + {b}) * {c}"),
                  ("What is {a} squared?", "{a} ** 2")],
        "test": [("Multiply {a} by {b} for me.", "{a} * {b}"),
                 ("If I add {a} and {b} and then multiply the total by {c}, what do I get?", "({a} + {b}) * {c}"),
                 ("Take {b} away from {a}. What's left?", "{a} - {b}"),
                 ("Split {a} into {b} equal parts; how big is each part?", "{a} / {b}"),
                 ("What do you get when you raise {a} to the power of 3?", "{a} ** 3")],
    },
    "convert": {
        "train": ["Convert {v} {fu} to {tu}.",
                  "How many {tu} is {v} {fu}?",
                  "What is {v} {fu} in {tu}?",
                  "{v} {fu} to {tu}"],
        "test": ["My recipe says {v} {fu}; how much is that in {tu}?",
                 "I ran {v} {fu} this morning, roughly how many {tu} is that?",
                 "Express {v} {fu} using {tu} instead.",
                 "Change {v} {fu} over to {tu}, please."],
    },
    "days": {
        "train": ["How many days are there between {d1} and {d2}?",
                  "Count the days from {d1} to {d2}.",
                  "Number of days from {d1} until {d2}?",
                  "How long is it in days from {d1} to {d2}?"],
        "test": ["My trip starts on {d1} and ends on {d2}. How many days is that?",
                 "If today is {d1}, how many days until {d2}?",
                 "Work out the gap in days between {d1} and {d2}."],
    },
    "weather_f": {
        "train": ["What's the temperature in {city} in Fahrenheit?",
                  "How warm is it in {city} in degrees Fahrenheit?",
                  "Tell me the temperature in {city} converted to Fahrenheit."],
        "test": ["My American friend wants to know how hot it is in {city}, in Fahrenheit.",
                 "Give me {city}'s current temperature but in °F."],
    },
    "stock_shares": {
        "train": ["How much would {n} shares of {ticker} cost?",
                  "What's the total cost of buying {n} shares of {ticker}?",
                  "I want {n} shares of {ticker}. How much will that be?"],
        "test": ["If I bought {n} {ticker} shares at today's price, what would I pay in total?",
                 "Price out a purchase of {n} shares of {ticker} for me."],
    },
}

# Unit pairs the user may ask about: (spoken from-unit, spoken to-unit, canonical from, canonical to)
UNIT_PAIRS = [
    ("kilometers", "miles", "km", "mi"), ("miles", "kilometers", "mi", "km"),
    ("kg", "pounds", "kg", "lb"), ("pounds", "kilograms", "lb", "kg"),
    ("meters", "feet", "m", "ft"), ("feet", "meters", "ft", "m"),
    ("liters", "gallons", "L", "gal"), ("gallons", "liters", "gal", "L"),
    ("cm", "inches", "cm", "in"), ("inches", "centimeters", "in", "cm"),
    ("degrees Celsius", "Fahrenheit", "C", "F"), ("degrees Fahrenheit", "Celsius", "F", "C"),
]

# No-tool requests: (query, answer used as training target, keyword the answer must contain or None)
NO_TOOL = {
    "train": [
        ("What is the capital of France?", "The capital of France is Paris.", "paris"),
        ("Who wrote Romeo and Juliet?", "Romeo and Juliet was written by William Shakespeare.", "shakespeare"),
        ("How many continents are there?", "There are 7 continents.", "7"),
        ("What is the chemical symbol for gold?", "The chemical symbol for gold is Au.", "au"),
        ("What planet is known as the Red Planet?", "Mars is known as the Red Planet.", "mars"),
        ("What is the largest ocean on Earth?", "The Pacific Ocean is the largest ocean on Earth.", "pacific"),
        ("How many legs does a spider have?", "A spider has 8 legs.", "8"),
        ("What language is spoken in Brazil?", "The main language spoken in Brazil is Portuguese.", "portuguese"),
        ("Who painted the Mona Lisa?", "The Mona Lisa was painted by Leonardo da Vinci.", "leonardo"),
        ("What gas do plants absorb from the air?", "Plants absorb carbon dioxide (CO2) from the air.", "carbon dioxide"),
        ("What is the boiling point of water at sea level in Celsius?", "Water boils at 100 degrees Celsius at sea level.", "100"),
        ("How many days are in a week?", "There are 7 days in a week.", "7"),
        ("What is the tallest mountain in the world?", "Mount Everest is the tallest mountain in the world.", "everest"),
        ("Translate 'thank you' into Spanish.", "'Thank you' in Spanish is 'gracias'.", "gracias"),
        ("What does CPU stand for?", "CPU stands for Central Processing Unit.", "central processing unit"),
        ("What is a stock market?", "A stock market is a place where shares of public companies are bought and sold.", "shares"),
        ("Why is the sky blue?", "The sky looks blue because air molecules scatter short blue wavelengths of sunlight more than other colours (Rayleigh scattering).", "scatter"),
        ("What causes rain?", "Rain forms when water vapour in clouds condenses into droplets that grow heavy enough to fall.", "water"),
        ("What is a kilometer?", "A kilometer is a unit of length equal to 1000 meters.", "1000"),
        ("Hi there!", "Hello! How can I help you today?", None),
        ("Thanks for your help.", "You're welcome! Let me know if you need anything else.", None),
        ("Write a two-line poem about the sea.", "The sea rolls in with silver light,\nand hums the shore to sleep at night.", None),
        ("Give me a tip for staying focused while studying.", "Try working in 25-minute blocks with short breaks in between, and keep your phone out of reach.", None),
        ("Tell me a fun fact about octopuses.", "Octopuses have three hearts and blue blood.", None),
        ("What's your name?", "I'm a helpful assistant. How can I help you?", None),
        ("Explain what an API is in one sentence.", "An API is a set of rules that lets one piece of software request data or actions from another.", None),
        ("Suggest a name for a pet cat.", "How about 'Mochi'?", None),
        ("What is the opposite of 'ancient'?", "The opposite of 'ancient' is 'modern'.", "modern"),
    ],
    "test": [
        ("Which country is home to the city of Kyoto?", "Kyoto is in Japan.", "japan"),
        ("Who developed the theory of relativity?", "Albert Einstein developed the theory of relativity.", "einstein"),
        ("What's the freezing point of water in Celsius?", "Water freezes at 0 degrees Celsius.", "0"),
        ("Name the longest river in Africa.", "The Nile is the longest river in Africa.", "nile"),
        ("How many sides does a hexagon have?", "A hexagon has 6 sides.", "6"),
        ("What does 'HTML' stand for?", "HTML stands for HyperText Markup Language.", "markup"),
        ("Which organ pumps blood around the human body?", "The heart pumps blood around the body.", "heart"),
        ("In general, why do stock prices go up and down?", "Stock prices move with supply and demand, driven by company results, news and investor expectations.", None),
        ("Explain in simple words what a weather forecast is.", "A weather forecast is a prediction of future weather made from current observations and models.", None),
        ("Good morning! How are you doing?", "Good morning! I'm doing well, thanks. How can I help?", None),
        ("Could you write me a short motivational sentence?", "Small steps every day add up to big results.", None),
        ("What is the plural of 'mouse'?", "The plural of 'mouse' is 'mice'.", "mice"),
        ("Recommend a good habit for better sleep.", "Keep a consistent bedtime and avoid screens for an hour before sleep.", None),
        ("What is the main ingredient in guacamole?", "The main ingredient in guacamole is avocado.", "avocado"),
        ("Tell me a short joke.", "Why did the scarecrow win an award? Because he was outstanding in his field.", None),
        ("Who was the first person to walk on the Moon?", "Neil Armstrong was the first person to walk on the Moon.", "armstrong"),
        ("Describe the colour of a ripe banana in one word.", "Yellow.", "yellow"),
        ("What do bees make?", "Bees make honey.", "honey"),
        ("Is a tomato a fruit or a vegetable, botanically?", "Botanically, a tomato is a fruit.", "fruit"),
        ("Say hello in French.", "Hello in French is 'bonjour'.", "bonjour"),
    ],
}

# ----------------------------------------------------------------------------
# Answer formatting (training targets)
# ----------------------------------------------------------------------------
MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August",
          "September", "October", "November", "December"]


def fmt_num(x):
    x = round(float(x), 2)
    return str(int(x)) if x == int(x) else f"{x:.2f}".rstrip("0").rstrip(".")


def random_date(rng):
    import datetime as dt
    d = dt.date(2024, 1, 1) + dt.timedelta(days=rng.randint(0, 900))
    return d


def spoken_date(d, style):
    if style == "iso":
        return d.isoformat()
    return f"{MONTHS[d.month - 1]} {d.day}, {d.year}"  # test uses written dates


def build(rng, category, split):
    """Return (query, gold_calls, checks, final_answer_text_fn) for one example."""
    tmpl = rng.choice(T[category][split]) if category in T else None

    if category == "weather":
        city = rng.choice(CITIES)
        q = tmpl.format(city=city if rng.random() > 0.15 else city.lower())
        calls = [("get_weather", {"city": city})]

        def answer(results):
            r = results[0]
            return rng.choice([
                f"It's currently {r['temperature_c']}°C and {r['condition']} in {r['city']}.",
                f"The weather in {r['city']} is {r['condition']} with a temperature of {r['temperature_c']}°C.",
            ])
        t, c = WEATHER[city.lower()]
        checks = [{"type": "num", "value": t}, {"type": "str", "value": c}]

    elif category == "stock":
        tk = rng.choice(TICKERS)
        q = tmpl.format(ticker=tk)
        calls = [("get_stock_price", {"ticker": tk})]

        def answer(results):
            r = results[0]
            return rng.choice([f"{r['ticker']} is trading at ${fmt_num(r['price_usd'])} per share.",
                               f"The current price of {r['ticker']} is ${fmt_num(r['price_usd'])}."])
        checks = [{"type": "num", "value": STOCKS[tk]}]

    elif category == "calc":
        q_t, e_t = tmpl
        a, b, c = rng.randint(12, 999), rng.randint(2, 99), rng.randint(2, 19)
        if "** 3" in e_t:
            a = rng.randint(2, 25)
        if "/" in e_t:
            a = b * rng.randint(2, 60) if rng.random() < 0.6 else a
        expr = e_t.format(a=a, b=b, c=c)
        q = q_t.format(a=a, b=b, c=c)
        calls = [("calculator", {"expression": expr})]
        value = execute_tool("calculator", {"expression": expr})["result"]

        def answer(results):
            return rng.choice([f"The result is {fmt_num(results[0]['result'])}.",
                               f"{expr} = {fmt_num(results[0]['result'])}."])
        checks = [{"type": "num", "value": value}]

    elif category == "convert":
        su, tu, cf, ct = rng.choice(UNIT_PAIRS)
        v = rng.choice([rng.randint(1, 500), round(rng.uniform(0.5, 120), 1)])
        q = tmpl.format(v=fmt_num(v), fu=su, tu=tu)
        calls = [("convert_units", {"value": v, "from_unit": cf, "to_unit": ct})]
        value = execute_tool("convert_units", calls[0][1])["result"]

        def answer(results):
            r = results[0]
            return f"{fmt_num(v)} {su} is about {fmt_num(r['result'])} {tu}."
        checks = [{"type": "num", "value": value}]

    elif category == "days":
        d1 = random_date(rng)
        import datetime as dt
        d2 = d1 + dt.timedelta(days=rng.randint(3, 400))
        style = "iso" if split == "train" else "written"
        q = tmpl.format(d1=spoken_date(d1, style), d2=spoken_date(d2, style))
        calls = [("days_between", {"start_date": d1.isoformat(), "end_date": d2.isoformat()})]

        def answer(results):
            return f"There are {results[0]['days']} days between {spoken_date(d1, style)} and {spoken_date(d2, style)}."
        checks = [{"type": "num", "value": (d2 - d1).days}]

    elif category == "weather_f":
        city = rng.choice(CITIES)
        q = tmpl.format(city=city)
        t, _ = WEATHER[city.lower()]
        calls = [("get_weather", {"city": city}),
                 ("convert_units", {"value": t, "from_unit": "C", "to_unit": "F"})]
        value = execute_tool("convert_units", calls[1][1])["result"]

        def answer(results):
            return f"It's {results[0]['temperature_c']}°C in {city}, which is about {fmt_num(results[1]['result'])}°F."
        checks = [{"type": "num", "value": value}]

    elif category == "stock_shares":
        tk, n = rng.choice(TICKERS), rng.randint(2, 60)
        q = tmpl.format(ticker=tk, n=n)
        calls = [("get_stock_price", {"ticker": tk}),
                 ("calculator", {"expression": f"{STOCKS[tk]} * {n}"})]
        value = execute_tool("calculator", calls[1][1])["result"]

        def answer(results):
            return (f"{tk} costs ${fmt_num(results[0]['price_usd'])} per share, so {n} shares "
                    f"would cost ${fmt_num(results[1]['result'])}.")
        checks = [{"type": "num", "value": value}]

    elif category == "no_tool":
        q, ans, kw = rng.choice(NO_TOOL[split])
        calls = []

        def answer(results):
            return ans
        checks = [{"type": "str", "value": kw}] if kw else []

    else:
        raise ValueError(category)

    return q, calls, checks, answer


def to_record(idx, split, category, q, calls, checks, answer):
    msgs = [{"role": "user", "content": q}]
    results = []
    for name, args in calls:  # one call per assistant turn (sequential chain)
        msgs.append({"role": "assistant", "content": "",
                     "tool_calls": [{"type": "function", "function": {"name": name, "arguments": args}}]})
        r = execute_tool(name, args)
        assert "error" not in r, (name, args, r)
        results.append(r)
        msgs.append({"role": "tool", "name": name, "content": tool_result_str(r)})
    msgs.append({"role": "assistant", "content": answer(results)})
    return {"id": f"{split}-{idx:04d}", "split": split, "category": category, "query": q,
            "gold_calls": [{"name": n, "arguments": a} for n, a in calls],
            "checks": checks, "messages": msgs}


# category -> count
TRAIN_MIX = {"weather": 110, "stock": 110, "calc": 110, "convert": 110, "days": 100,
             "weather_f": 50, "stock_shares": 50, "no_tool": 160}            # = 800
TEST_MIX = {"weather": 12, "stock": 12, "calc": 12, "convert": 12, "days": 12,
            "weather_f": 10, "stock_shares": 10, "no_tool": 20}               # = 100


def make_split(split, mix, seed):
    rng = random.Random(seed)
    recs, seen = [], set()
    plan = [c for c, n in mix.items() for _ in range(n)]
    rng.shuffle(plan)
    for cat in plan:
        for _ in range(50):  # avoid exact duplicate queries where possible
            q, calls, checks, answer = build(rng, cat, split)
            if q not in seen or cat == "no_tool":
                break
        seen.add(q)
        recs.append(to_record(len(recs), split, cat, q, calls, checks, answer))
    return recs


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default=os.path.join(os.path.dirname(__file__), "data"))
    ap.add_argument("--seed", type=int, default=SEED)
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    for split, mix, s in [("train", TRAIN_MIX, args.seed), ("test", TEST_MIX, args.seed + 1)]:
        recs = make_split(split, mix, s)
        with open(os.path.join(args.out, f"{split}.jsonl"), "w") as f:
            for r in recs:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"{split}: {len(recs)} examples, {sum(not r['gold_calls'] for r in recs)} need no tool")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile agent.py
"""Minimal tool-calling agent: generate -> parse tool call -> execute -> feed back -> repeat.

Uses the model's own chat template (Qwen2.5 style: tools in the system prompt,
calls emitted as <tool_call>{"name": ..., "arguments": {...}}</tool_call>).
`run_agent_batch` runs many independent conversations in lock-step so the GPU/CPU
generates for all active conversations in one batch.
"""
import json
import re

from tools import TOOL_SCHEMAS, TOOLS, execute_tool, tool_result_str

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

SYSTEM_PROMPT = (
    "You are a helpful assistant with access to tools. "
    "Call a tool only when the request needs live data or a calculation the tools provide; "
    "otherwise answer directly. After receiving tool results, answer the user in plain text."
)

MAX_STEPS = 4
_TOOL_CALL_RE = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.S)
_STRIP_TOKENS = ["<|im_end|>", "<|endoftext|>", "<|im_start|>"]


def load_model(model_id=MODEL_ID, adapter_path=None, device=None, dtype=None):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    dtype = dtype or (torch.float16 if device == "cuda" else torch.float32)
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype).to(device)
    if adapter_path:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter_path)
        model = model.merge_and_unload()
    model.eval()
    return model, tok


# ----------------------------------------------------------------------------
# Parsing
# ----------------------------------------------------------------------------
def parse_tool_calls(text):
    """Return (attempted, valid, calls).

    attempted: the output looks like an attempt to call a tool (tagged or raw JSON).
    valid:     every call parses as JSON in the expected <tool_call> format, names a
               known tool, and has an object of arguments.
    calls:     list of {"name", "arguments"} (only when valid).
    """
    blocks = _TOOL_CALL_RE.findall(text)
    stripped = text.strip()
    raw_json_attempt = (stripped.startswith("{") and '"name"' in stripped) or "<tool_call>" in text
    if not blocks:
        return (raw_json_attempt, False, []) if raw_json_attempt else (False, False, [])
    calls = []
    for b in blocks:
        try:
            obj = json.loads(b)
        except json.JSONDecodeError:
            return True, False, []
        if not isinstance(obj, dict):
            return True, False, []
        name, args = obj.get("name"), obj.get("arguments", obj.get("parameters"))
        if isinstance(args, str):
            try:
                args = json.loads(args)
            except json.JSONDecodeError:
                return True, False, []
        if name not in TOOLS or not isinstance(args, dict):
            return True, False, [{"name": name, "arguments": args}]
        calls.append({"name": name, "arguments": args})
    return True, True, calls


def clean(text):
    for t in _STRIP_TOKENS:
        text = text.replace(t, "")
    return text.strip()


# ----------------------------------------------------------------------------
# Generation
# ----------------------------------------------------------------------------
def generate_batch(model, tok, conversations, max_new_tokens=200):
    import torch
    prompts = [tok.apply_chat_template(c, tools=TOOL_SCHEMAS, add_generation_prompt=True, tokenize=False)
               for c in conversations]
    enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id, temperature=None, top_p=None, top_k=None)
    new = out[:, enc["input_ids"].shape[1]:]
    return [clean(tok.decode(row, skip_special_tokens=False)) for row in new]


def run_agent_batch(model, tok, queries, system_prompt=SYSTEM_PROMPT, max_steps=MAX_STEPS,
                    batch_size=16, max_new_tokens=200):
    """Run the agent loop on every query. Returns one trace dict per query."""
    states = []
    for q in queries:
        states.append({
            "query": q,
            "messages": [{"role": "system", "content": system_prompt}, {"role": "user", "content": q}],
            "steps": [],          # one entry per model turn
            "final": None,        # plain-text final answer (None if never reached)
            "stop": None,         # "answer" | "invalid_call" | "max_steps"
            "done": False,
        })
    for _ in range(max_steps):
        active = [s for s in states if not s["done"]]
        if not active:
            break
        for i in range(0, len(active), batch_size):
            chunk = active[i:i + batch_size]
            outs = generate_batch(model, tok, [s["messages"] for s in chunk], max_new_tokens)
            for s, text in zip(chunk, outs):
                attempted, valid, calls = parse_tool_calls(text)
                step = {"raw": text, "attempted": attempted, "valid": valid, "calls": calls, "results": []}
                s["steps"].append(step)
                if not attempted:
                    s["final"], s["stop"], s["done"] = text, "answer", True
                    s["messages"].append({"role": "assistant", "content": text})
                    continue
                if not valid:
                    s["stop"], s["done"] = "invalid_call", True
                    continue
                s["messages"].append({"role": "assistant", "content": "", "tool_calls": [
                    {"type": "function", "function": {"name": c["name"], "arguments": c["arguments"]}}
                    for c in calls]})
                for c in calls:
                    r = execute_tool(c["name"], c["arguments"])
                    step["results"].append(r)
                    s["messages"].append({"role": "tool", "name": c["name"], "content": tool_result_str(r)})
    for s in states:
        if not s["done"]:
            s["stop"], s["done"] = "max_steps", True
    return [{k: v for k, v in s.items() if k != "done"} for s in states]


def chat(model, tok, query, **kw):
    """Convenience: run one query and print the trace."""
    tr = run_agent_batch(model, tok, [query], **kw)[0]
    for i, st in enumerate(tr["steps"]):
        print(f"[step {i}] {st['raw']}")
        for r in st["results"]:
            print(f"   -> tool result: {r}")
    print(f"FINAL ({tr['stop']}): {tr['final']}")
    return tr

In [ ]:
%%writefile evaluate.py
"""Metrics and failure analysis for agent traces.

Metrics (computed per test example, then averaged over the relevant subset):
  call_validity : tool-needed examples whose FIRST model turn parses as a tool call
                  in the expected <tool_call>{"name": ..., "arguments": {...}}</tool_call>
                  format (format only; calling a non-existent tool counts as wrong_tool).
  call_accuracy : tool-needed examples whose full sequence of calls matches the gold
                  sequence (right tools, right arguments, right order).
  abstention    : no-tool examples where the model answered without calling a tool.
  task_success  : all examples; final plain-text answer satisfies every gold check
                  (and, for no-tool examples, no tool was called).
"""
import argparse
import json
import math
import re
from collections import Counter

from tools import TOOLS, canon_unit, eval_expression, parse_date

_NUM_RE = re.compile(r"-?\d[\d,]*\.?\d*")


# ----------------------------------------------------------------------------
# Argument normalisation / call matching
# ----------------------------------------------------------------------------
def _num(x):
    try:
        return float(str(x).replace(",", ""))
    except (TypeError, ValueError):
        return None


def _close(a, b, rel=2e-3, abs_=0.5):
    return a is not None and b is not None and math.isclose(a, b, rel_tol=rel, abs_tol=abs_)


def args_match(name, pred, gold):
    if not isinstance(pred, dict):
        return False
    if name == "calculator":
        try:
            return _close(eval_expression(pred.get("expression")), eval_expression(gold["expression"]), rel=1e-4)
        except Exception:  # noqa: BLE001
            return False
    if name == "convert_units":
        return (_close(_num(pred.get("value")), _num(gold["value"]), rel=1e-4)
                and canon_unit(pred.get("from_unit")) == canon_unit(gold["from_unit"])
                and canon_unit(pred.get("to_unit")) == canon_unit(gold["to_unit"]))
    if name == "get_weather":
        return str(pred.get("city", "")).strip().lower() == gold["city"].lower()
    if name == "get_stock_price":
        return str(pred.get("ticker", "")).strip().upper() == gold["ticker"].upper()
    if name == "days_between":
        return (parse_date(pred.get("start_date", "")) == parse_date(gold["start_date"])
                and parse_date(pred.get("end_date", "")) == parse_date(gold["end_date"]))
    return pred == gold


def pred_calls(trace):
    return [c for st in trace["steps"] if st["valid"] for c in st["calls"]]


def format_ok(st):
    """Turn parses as <tool_call>{json with name + object arguments}</tool_call>, whatever the tool name."""
    return bool(st["attempted"] and (st["valid"] or any(
        isinstance(c.get("arguments"), dict) for c in st["calls"])))


def hallucinated_tool(st):
    return bool(st["attempted"] and not st["valid"] and any(c.get("name") not in TOOLS for c in st["calls"]))


def attempted_calls(trace):
    """Every parsed call in order, including calls to tools that do not exist."""
    return [c for st in trace["steps"] if st["valid"] or hallucinated_tool(st) for c in st["calls"]]


def _stem(w):
    return w if len(w) <= 4 else w[:max(4, len(w) - 3)]


def str_ok(value, text):
    """Keyword check robust to inflection: 'drizzle' matches 'drizzling', 'snow' matches 'snowing'."""
    value, text = value.lower(), text.lower()
    if value in text:
        return True
    words = [w for w in re.findall(r"[a-z0-9]+", value) if w not in ("and", "the", "of")]
    return bool(words) and all(re.search(r"\b" + re.escape(_stem(w)), text) for w in words)


def calls_match(pred, gold):
    return len(pred) == len(gold) and all(
        p["name"] == g["name"] and args_match(g["name"], p["arguments"], g["arguments"])
        for p, g in zip(pred, gold))


def answer_ok(final, checks):
    if final is None:
        return False
    low = final.lower()
    nums = [_num(m) for m in _NUM_RE.findall(final)]
    for c in checks:
        if c["type"] == "str" and not str_ok(c["value"], low):
            return False
        if c["type"] == "num" and not any(_close(n, float(c["value"])) for n in nums):
            return False
    return True


# ----------------------------------------------------------------------------
# Per-example scoring + failure category
# ----------------------------------------------------------------------------
def score_example(ex, tr):
    gold = ex["gold_calls"]
    needs_tool = bool(gold)
    first = tr["steps"][0] if tr["steps"] else {"attempted": False, "valid": False, "calls": []}
    any_attempt = any(st["attempted"] for st in tr["steps"])
    pcalls = pred_calls(tr)
    r = {"id": ex["id"], "category": ex["category"], "needs_tool": needs_tool}
    if needs_tool:
        r["valid"] = format_ok(first)
        r["call_acc"] = calls_match(pcalls, gold)
        r["success"] = answer_ok(tr["final"], ex["checks"])
    else:
        r["abstain"] = not any_attempt
        r["success"] = r["abstain"] and answer_ok(tr["final"], ex["checks"])
    r["failure"] = None if r["success"] else categorize(ex, tr, pcalls)
    return r


FAILURE_CATEGORIES = [
    "missed_call",        # tool needed but model answered directly (often hallucinated a value)
    "wrong_format",       # tried to call a tool but output does not parse in the expected format
    "wrong_tool",         # parsable call to the wrong tool (or to a tool that does not exist)
    "wrong_argument",     # right tool, wrong arguments
    "incomplete_chain",   # multi-step task: stopped after the first call
    "extra_calls",        # repeated / unnecessary extra calls, often hitting the step limit
    "ignored_result",     # calls correct but final answer does not use the tool result
    "unnecessary_call",   # no tool needed but model called one
    "wrong_answer",       # no tool needed, no call, but answer is wrong
]


def categorize(ex, tr, pcalls):
    gold = ex["gold_calls"]
    any_attempt = any(st["attempted"] for st in tr["steps"])
    any_invalid = any(st["attempted"] and not st["valid"] for st in tr["steps"])
    if not gold:
        return "unnecessary_call" if any_attempt else "wrong_answer"
    if not any_attempt:
        return "missed_call"
    pcalls = attempted_calls(tr)
    if any_invalid and not pcalls:
        return "wrong_format"
    # compare call by call
    for i, g in enumerate(gold):
        if i >= len(pcalls):
            return "wrong_format" if any_invalid else "incomplete_chain"
        p = pcalls[i]
        if p["name"] != g["name"]:
            return "wrong_tool"
        if not args_match(g["name"], p["arguments"], g["arguments"]):
            return "wrong_argument"
    if len(pcalls) > len(gold):
        return "extra_calls"
    if any(st["attempted"] and not format_ok(st) for st in tr["steps"]):
        return "wrong_format"
    return "ignored_result"


def summarize(rows):
    tool = [r for r in rows if r["needs_tool"]]
    notool = [r for r in rows if not r["needs_tool"]]
    pct = lambda xs, k: 100.0 * sum(x[k] for x in xs) / max(len(xs), 1)  # noqa: E731
    return {
        "n_total": len(rows), "n_tool": len(tool), "n_no_tool": len(notool),
        "call_validity": pct(tool, "valid"),
        "call_accuracy": pct(tool, "call_acc"),
        "abstention": pct(notool, "abstain"),
        "task_success": pct(rows, "success"),
        "task_success_tool": pct(tool, "success"),
        "task_success_multi": pct([r for r in tool if r["category"] in ("weather_f", "stock_shares")], "success"),
        "failures": dict(Counter(r["failure"] for r in rows if r["failure"])),
    }


def evaluate(test, traces):
    rows = [score_example(ex, tr) for ex, tr in zip(test, traces)]
    return rows, summarize(rows)


def compare(rows_base, rows_lora):
    """Transitions per example between baseline and LoRA (fixed / persisted / introduced)."""
    out = {"fixed": Counter(), "persisted": Counter(), "introduced": Counter()}
    for b, l in zip(rows_base, rows_lora):
        if b["failure"] and not l["failure"]:
            out["fixed"][b["failure"]] += 1
        elif b["failure"] and l["failure"]:
            out["persisted"][f"{b['failure']} -> {l['failure']}"] += 1
        elif l["failure"] and not b["failure"]:
            out["introduced"][l["failure"]] += 1
    return {k: dict(v) for k, v in out.items()}


def print_table(summaries):
    keys = ["call_validity", "call_accuracy", "abstention", "task_success", "task_success_tool", "task_success_multi"]
    names = list(summaries)
    print(f"{'metric':<22}" + "".join(f"{n:>12}" for n in names))
    for k in keys:
        print(f"{k:<22}" + "".join(f"{summaries[n][k]:>11.1f}%" for n in names))
    print()
    cats = sorted({c for s in summaries.values() for c in s["failures"]}, key=FAILURE_CATEGORIES.index)
    print(f"{'failure category':<22}" + "".join(f"{n:>12}" for n in names))
    for c in cats:
        print(f"{c:<22}" + "".join(f"{summaries[n]['failures'].get(c, 0):>12}" for n in names))


def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--test", required=True)
    ap.add_argument("--traces", nargs="+", required=True, help="name=path.jsonl")
    a = ap.parse_args()
    test = load_jsonl(a.test)
    sums, rows = {}, {}
    for spec in a.traces:
        name, path = spec.split("=")
        rows[name], sums[name] = evaluate(test, load_jsonl(path))
    print_table(sums)
    if len(rows) == 2:
        n1, n2 = list(rows)
        print(json.dumps(compare(rows[n1], rows[n2]), indent=2))

In [ ]:
%%writefile train.py
"""LoRA finetuning with loss on assistant turns only.

The full gold conversation (system + tools, user, assistant tool-call turns, tool
results, final assistant answer) is rendered with the model's chat template.
Labels are -100 everywhere except tokens inside assistant turns
(`<|im_start|>assistant\n ... <|im_end|>`), found via the tokenizer offset mapping.
"""
import argparse
import json
import math
import os
import random
import re
import time

import torch

from agent import MODEL_ID, SYSTEM_PROMPT
from tools import TOOL_SCHEMAS

_ASSIST_RE = re.compile(r"<\|im_start\|>assistant\n(.*?<\|im_end\|>)", re.S)


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def encode(tok, messages, max_len):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}] + messages
    text = tok.apply_chat_template(msgs, tools=TOOL_SCHEMAS, tokenize=False)
    enc = tok(text, return_offsets_mapping=True, add_special_tokens=False, truncation=True, max_length=max_len)
    ids, offs = enc["input_ids"], enc["offset_mapping"]
    spans = [m.span(1) for m in _ASSIST_RE.finditer(text)]
    labels = [-100] * len(ids)
    for i, (a, b) in enumerate(offs):
        if b > a and any(s <= a and b <= e for s, e in spans):
            labels[i] = ids[i]
    return {"input_ids": ids, "labels": labels}


def collate(batch, pad_id):
    L = max(len(x["input_ids"]) for x in batch)
    ids = torch.full((len(batch), L), pad_id, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    for i, x in enumerate(batch):  # right padding for training
        n = len(x["input_ids"])
        ids[i, :n] = torch.tensor(x["input_ids"])
        lab[i, :n] = torch.tensor(x["labels"])
        att[i, :n] = 1
    return {"input_ids": ids, "labels": lab, "attention_mask": att}


def show_masking(tok, example):
    """Print which part of one example carries loss (sanity check)."""
    ids, lab = example["input_ids"], example["labels"]
    kept = tok.decode([t for t, l in zip(ids, lab) if l != -100])
    print(f"tokens: {len(ids)}, supervised: {sum(l != -100 for l in lab)}")
    print("supervised text:\n" + kept)


def train(train_path, out_dir, model_id=MODEL_ID, epochs=2, lr=2e-4, batch_size=8, micro_batch=4,
          r=16, alpha=32, dropout=0.05, max_len=1024, seed=42, max_steps=None, log_every=10):
    from peft import LoraConfig, get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup

    set_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    use_amp = device == "cuda"  # fp16 autocast on T4 (no bf16), fp32 on CPU
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32).to(device)
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, task_type="CAUSAL_LM",
                     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(model, cfg)
    model.print_trainable_parameters()

    with open(train_path) as f:
        data = [encode(tok, json.loads(l)["messages"], max_len) for l in f]
    show_masking(tok, data[0])

    accum = max(batch_size // micro_batch, 1)
    steps_per_epoch = math.ceil(len(data) / batch_size)
    total = steps_per_epoch * epochs if max_steps is None else max_steps
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.0)
    sched = get_cosine_schedule_with_warmup(opt, int(0.05 * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    model.train()
    step, t0, rng, log = 0, time.time(), random.Random(seed), []
    for ep in range(epochs):
        order = list(range(len(data)))
        rng.shuffle(order)
        for s in range(0, len(order), batch_size):
            if step >= total:
                break
            idx = order[s:s + batch_size]
            loss_sum = 0.0
            for m in range(0, len(idx), micro_batch):
                b = collate([data[i] for i in idx[m:m + micro_batch]], tok.pad_token_id)
                b = {k: v.to(device) for k, v in b.items()}
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                    loss = model(**b).loss / accum
                scaler.scale(loss).backward()
                loss_sum += loss.item()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            sched.step()
            step += 1
            log.append({"step": step, "epoch": ep, "loss": loss_sum, "lr": sched.get_last_lr()[0]})
            if step % log_every == 0 or step == 1:
                print(f"step {step}/{total} epoch {ep} loss {loss_sum:.4f} "
                      f"lr {sched.get_last_lr()[0]:.2e} {time.time() - t0:.0f}s", flush=True)
    os.makedirs(out_dir, exist_ok=True)
    model.save_pretrained(out_dir)
    with open(os.path.join(out_dir, "train_log.json"), "w") as f:
        json.dump({"config": dict(model_id=model_id, epochs=epochs, lr=lr, batch_size=batch_size, r=r,
                                  alpha=alpha, dropout=dropout, seed=seed, n_train=len(data), steps=step,
                                  seconds=time.time() - t0), "log": log}, f, indent=1)
    print(f"saved adapter to {out_dir} after {step} steps, {time.time() - t0:.0f}s")
    return out_dir


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--train", default=os.path.join(os.path.dirname(__file__), "data", "train.jsonl"))
    ap.add_argument("--out", default=os.path.join(os.path.dirname(__file__), "outputs", "lora_adapter"))
    ap.add_argument("--epochs", type=int, default=2)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--batch_size", type=int, default=8)
    ap.add_argument("--micro_batch", type=int, default=4)
    ap.add_argument("--r", type=int, default=16)
    ap.add_argument("--max_steps", type=int, default=None)
    ap.add_argument("--seed", type=int, default=42)
    a = ap.parse_args()
    train(a.train, a.out, epochs=a.epochs, lr=a.lr, batch_size=a.batch_size, micro_batch=a.micro_batch,
          r=a.r, max_steps=a.max_steps, seed=a.seed)

In [ ]:
%%writefile run_eval.py
"""Run the agent on the test set (baseline or with a LoRA adapter) and save traces + metrics."""
import argparse
import json
import os
import time

import torch

from agent import MODEL_ID, SYSTEM_PROMPT, load_model, run_agent_batch
from evaluate import evaluate, load_jsonl


def run(name, test_path, out_dir, adapter=None, batch_size=16, limit=None, system_prompt=SYSTEM_PROMPT):
    torch.manual_seed(42)
    test = load_jsonl(test_path)[:limit]
    model, tok = load_model(MODEL_ID, adapter_path=adapter)
    t0 = time.time()
    traces = run_agent_batch(model, tok, [ex["query"] for ex in test], system_prompt=system_prompt,
                             batch_size=batch_size)
    rows, summary = evaluate(test, traces)
    summary["seconds"] = time.time() - t0
    summary["system_prompt"] = system_prompt
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, f"traces_{name}.jsonl"), "w") as f:
        for ex, tr, r in zip(test, traces, rows):
            tr = {k: v for k, v in tr.items() if k != "messages"}
            f.write(json.dumps({"id": ex["id"], "category": ex["category"], **tr, "score": r}, ensure_ascii=False) + "\n")
    with open(os.path.join(out_dir, f"summary_{name}.json"), "w") as f:
        json.dump(summary, f, indent=1)
    print(name, json.dumps(summary, indent=1))
    return traces, rows, summary


if __name__ == "__main__":
    here = os.path.dirname(os.path.abspath(__file__))
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", required=True)
    ap.add_argument("--adapter", default=None)
    ap.add_argument("--test", default=os.path.join(here, "data", "test.jsonl"))
    ap.add_argument("--out", default=os.path.join(here, "outputs"))
    ap.add_argument("--batch_size", type=int, default=16)
    ap.add_argument("--limit", type=int, default=None)
    a = ap.parse_args()
    run(a.name, a.test, a.out, a.adapter, a.batch_size, a.limit)

## 2. Data
Templates for train and test never overlap; the date task uses ISO dates in train and written dates (`March 1, 2024`) in test.

In [ ]:
!python make_data.py --out data --seed 42
from evaluate import load_jsonl
from collections import Counter
train, test = load_jsonl('data/train.jsonl'), load_jsonl('data/test.jsonl')
print('train', Counter(r['category'] for r in train))
print('test ', Counter(r['category'] for r in test))
for r in test[:5]:
    print('-', r['query'], '=>', [c['name'] for c in r['gold_calls']] or 'NO TOOL')

## 3. The agent loop on the untouched model (demo)

In [ ]:
import torch, json
from agent import load_model, chat, SYSTEM_PROMPT, MODEL_ID
from tools import TOOL_SCHEMAS
model, tok = load_model(MODEL_ID)
print(tok.apply_chat_template([{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':'weather in Oslo?'}],
                              tools=TOOL_SCHEMAS, add_generation_prompt=True, tokenize=False)[-600:])
for q in ['How much would 12 shares of ZORB cost?', 'What is the capital of Japan?',
          'Count the days from March 3, 2024 to May 1, 2024.']:
    print('='*80); print('Q:', q); chat(model, tok, q)
del model; torch.cuda.empty_cache()

## 4. Baseline evaluation

In [ ]:
from run_eval import run
base_traces, base_rows, base_sum = run('base', 'data/test.jsonl', OUT, adapter=None, batch_size=16)

## 5. LoRA finetuning (loss on assistant turns only)
r=16, alpha=32, dropout 0.05, all linear layers, lr 2e-4 cosine, 2 epochs, batch 8, fp16 autocast.

In [ ]:
from train import train, encode
from transformers import AutoTokenizer
import os
ADAPTER = os.path.join(OUT, 'lora_adapter')
if os.path.exists(os.path.join(ADAPTER, 'adapter_config.json')):
    print('adapter already trained, skipping (delete the folder to retrain)')
else:
    train('data/train.jsonl', ADAPTER, epochs=2, lr=2e-4, batch_size=8, micro_batch=4, r=16, alpha=32, seed=SEED)

In [ ]:
import json, matplotlib.pyplot as plt
log = json.load(open(os.path.join(ADAPTER, 'train_log.json')))
print(log['config'])
plt.figure(figsize=(5,3)); plt.plot([l['step'] for l in log['log']], [l['loss'] for l in log['log']])
plt.xlabel('step'); plt.ylabel('train loss (assistant tokens)'); plt.tight_layout()
plt.savefig(os.path.join(OUT, 'train_loss.png'), dpi=150); plt.show()

## 6. LoRA evaluation (same test set, same system prompt)

In [ ]:
lora_traces, lora_rows, lora_sum = run('lora', 'data/test.jsonl', OUT, adapter=ADAPTER, batch_size=16)

## 7. Baseline vs LoRA

In [ ]:
from evaluate import print_table, compare
print_table({'base': base_sum, 'lora': lora_sum})
trans = compare(base_rows, lora_rows)
print(json.dumps(trans, indent=1))
json.dump({'base': base_sum, 'lora': lora_sum, 'transitions': trans}, open(os.path.join(OUT, 'results.json'), 'w'), indent=1)

## 8. Failure analysis
All failures of each model, with the raw model turns. Categories are assigned automatically and then checked by hand.

In [ ]:
def show_failures(name, traces, rows, k=15):
    fails = [(ex, tr, r) for ex, tr, r in zip(test, traces, rows) if r['failure']]
    print(f'##### {name}: {len(fails)} failures (showing {min(k, len(fails))})')
    for ex, tr, r in fails[:k]:
        print(f"\n[{r['failure']}] {ex['id']} ({ex['category']}) Q: {ex['query']}")
        print('   gold calls:', [(c['name'], c['arguments']) for c in ex['gold_calls']] or 'none')
        for i, st in enumerate(tr['steps']):
            print(f'   turn {i}:', st['raw'][:300].replace('\n', ' '))
        print('   final:', (tr['final'] or '')[:200], '| stop:', tr['stop'])
show_failures('BASE', base_traces, base_rows)

In [ ]:
show_failures('LORA', lora_traces, lora_rows)

### Notes for the write-up
* Which categories did LoRA fix / leave / introduce? → see `transitions` above.
* Could a better prompt alone have been enough? Try the cell below: a stronger system prompt with a format example, on the base model.

In [ ]:
STRONG_PROMPT = SYSTEM_PROMPT + (
    ' To call a tool, reply ONLY with <tool_call>{"name": <tool>, "arguments": {...}}</tool_call>. '
    'Never guess weather, stock prices, conversions, date differences or arithmetic: use the tool. '
    'Call one tool at a time; when you have the result, give the final answer with the number from the result. '
    'Dates must be YYYY-MM-DD. For questions that need no tool, answer directly without any tool call.')
pt_traces, pt_rows, pt_sum = run('base_prompt', 'data/test.jsonl', OUT, adapter=None, batch_size=16,
                                 system_prompt=STRONG_PROMPT)
print_table({'base': base_sum, 'base+prompt': pt_sum, 'lora': lora_sum})
json.dump({'base': base_sum, 'base_strong_prompt': pt_sum, 'lora': lora_sum, 'transitions': trans},
          open(os.path.join(OUT, 'results.json'), 'w'), indent=1)